# Research questions

Pick **one** of the eight questions below and answer it experimentally in two to five days.
Each asks whether an idea from the literature buys quality per FLOP at the scale of this
project — a 108-feature table and a small book, in NumPy, on a CPU — which no paper has
measured.

| baseline | score | cost |
|---|---|---|
| q05 Adult MLP (108→64→ReLU→2) | test accuracy **0.8543** | 84.23 GFLOP train · 232 MFLOP inference |
| always answer `<=50K` | test accuracy **0.7592** | 0 |
| q06 word2vec on *Flatland* (D=32, K=5) | silhouette **−0.058 → +0.141** | 6.6 GFLOP train |

1. Read the papers of your question.
2. Reproduce the baseline with `research/benchmark.py` before changing anything.
3. Change one thing, measure score **and** FLOPs, write the report.

## Protocol

- **Seeds.** At least three (`seeds=(0, 1, 2)`); five for the embedding questions, whose
  silhouette is noisy. Report mean ± standard deviation.
- **Error bars.** A difference smaller than one standard deviation is not a result.
- **Cost.** Every score comes with its FLOPs — training FLOPs for questions about learning,
  inference FLOPs for questions about using the model.
- **What is fixed.** State whether you fixed the number of epochs or the FLOP budget.
- **The counter.** Exact for matmuls, approximate for elementwise ops, blind to zeros and
  ternary weights. If your question depends on that, count it yourself and say how.

In [ ]:
# Run me first: make the project importable whether Jupyter started in the project
# root or inside research/.
import pathlib
import sys

ROOT = pathlib.Path.cwd()
if not (ROOT / "dlcb").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np

from dlcb import engine as cpu
from dlcb import nn
from dlcb import optim
from dlcb.loss import cross_entropy
from research.benchmark import (MLP, baseline_mlp, count_flops, pareto, repeat,
                                run_adult, run_sgns, summarize)

print("ready — project root:", ROOT)

### The instrument

`research/benchmark.py` returns a score together with its FLOPs: `run_adult` (q05) and
`run_sgns` (q06). `repeat` adds seeds, `summarize` builds the table, `pareto` draws the
plot. The hooks you will need: `loss_fn`, `subset`, `on_epoch` for Adult; `extra_loss`,
`neg_k` for the embeddings. Every answer has the shape of the cell below (a minute or two).

In [ ]:
rows = []
rows += repeat(lambda s: run_adult(baseline_mlp, seed=s, name="baseline"))
rows += repeat(lambda s: run_adult(lambda n: MLP(n, hidden=32), seed=s, name="hidden=32"))

print(summarize(rows, baseline="baseline"))

In [ ]:
# The same rows as a picture: score against what it cost.
pareto(rows, cost="infer_flops", title="Adult: accuracy vs inference cost")

---

## The questions

| # | question | data | you implement |
|---|---|---|---|
| 1 | [Ternary weights](#q1) | Adult | a ternary op with a straight-through backward |
| 2 | [Distillation](#q2) | Adult | a soft-target `loss_fn` |
| 3 | [Data pruning](#q3) | Adult | EL2N scores and `subset` |
| 4 | [Orthogonality regularisation](#q4) | Flatland | `extra_loss` = λ‖WᵀW − I‖² |
| 5 | [Isotropy vs. clusters](#q5) | Flatland | an isotropy penalty |
| 6 | [Training without negatives](#q6) | Flatland | a dimension regulariser with `neg_k=0` |
| 7 | [Zipfian whitening](#q7) | Flatland | frequency-weighted whitening |
| 8 | [A linear gender direction](#q8) | Flatland | difference vectors and projections |

<a id="q1"></a>
## 1. Ternary weights

**Question.** How much accuracy is lost when the first layer's weights are restricted to {−1, 0, +1} with one scale per matrix — and does a wider layer buy it back?

**Read**
- Ma et al., *The Era of 1-bit LLMs: All Large Language Models are in 1.58 Bits*, 2024 — [arXiv:2402.17764](https://arxiv.org/abs/2402.17764)
- Bengio, Léonard & Courville, *Estimating or Propagating Gradients Through Stochastic Neurons*, 2013 — [arXiv:1308.3432](https://arxiv.org/abs/1308.3432)

**Measure**
- A `ternary` op: absmean quantisation forward, identity backward (straight-through). Gradient-check it.
- Test accuracy for `hidden` ∈ {64, 128, 256}, ternary and full precision.
- Effective inference cost: a ternary matmul needs only additions. Count them yourself.

<a id="q2"></a>
## 2. Distillation

**Question.** Does a small student (`hidden=8`) trained on a wide teacher's soft probabilities beat the same student trained on the labels alone?

**Read**
- Hinton, Vinyals & Dean, *Distilling the Knowledge in a Neural Network*, 2015 — [arXiv:1503.02531](https://arxiv.org/abs/1503.02531)
- Beyer et al., *Knowledge Distillation: A Good Teacher is Patient and Consistent*, CVPR 2022 — [arXiv:2106.05237](https://arxiv.org/abs/2106.05237)

**Measure**
- Student accuracy over temperature T ∈ {1, 2, 4, 8} and mixing weight α, through `loss_fn`.
- The teacher's training FLOPs belong to the student's cost.
- Does a longer schedule (Beyer's *patient* teacher) change the answer?

<a id="q3"></a>
## 3. Data pruning

**Question.** Can you train on half of the rows without losing accuracy — and should you keep the easy rows or the hard ones?

**Read**
- Sorscher, Geirhos, Shekhar, Ganguli & Morcos, *Beyond Neural Scaling Laws: Beating Power Law Scaling via Data Pruning*, NeurIPS 2022 — [arXiv:2206.14486](https://arxiv.org/abs/2206.14486)
- Paul, Ganguli & Dziugaite, *Deep Learning on a Data Diet*, NeurIPS 2021 — [arXiv:2107.07075](https://arxiv.org/abs/2107.07075)

**Measure**
- EL2N scores after a few epochs; kept fractions {0.1, 0.3, 0.5, 0.7, 1.0} via `subset`.
- Keep-hard vs. keep-easy vs. random at each fraction; scoring FLOPs count.
- Sorscher et al. predict keep-easy wins when data is scarce and keep-hard when it is abundant. Check.

<a id="q4"></a>
## 4. Orthogonality regularisation

**Question.** Does adding λ‖WᵀW − I‖²_F on the input table prevent dimensional collapse and raise the silhouette?

**Read**
- He, Du & Ma, *Preventing Dimensional Collapse in Self-Supervised Learning via Orthogonality Regularization*, NeurIPS 2024 — [arXiv:2411.00392](https://arxiv.org/abs/2411.00392)

**Measure**
- Effective rank of the table — exp of the entropy of its normalised singular values — with and without the penalty.
- Silhouette for λ ∈ {0, 1e-4, 1e-3, 1e-2, 1e-1}, through `extra_loss`.
- FLOPs of the penalty next to those of the SGNS loss.

<a id="q5"></a>
## 5. Isotropy vs. clusters

**Question.** Does pushing the embeddings towards isotropy during training destroy the word groups the silhouette measures?

**Read**
- Mickus, Grönroos & Attieh, *Isotropy, Clusters, and Classifiers*, ACL 2024 — [arXiv:2402.03191](https://arxiv.org/abs/2402.03191)

**Measure**
- An isotropy score of the table, e.g. smallest / largest eigenvalue of its covariance.
- A differentiable penalty, e.g. ‖C − (tr C / D) I‖²_F with C the covariance, swept in λ via `extra_loss`.
- Silhouette against isotropy on one plot. Compare with the prediction of question 7.

<a id="q6"></a>
## 6. Training without negatives

**Question.** Can a dimension regulariser replace negative sampling — the same silhouette for fewer training FLOPs?

**Read**
- Liu, Seshadri, Eliassi-Rad & Ugander, *Bypassing Skip-Gram Negative Sampling: Dimension Regularization as a More Efficient Alternative for Graph Embeddings*, KDD 2025 — [arXiv:2405.00172](https://arxiv.org/abs/2405.00172)

**Measure**
- `neg_k=0` plus their regulariser through `extra_loss`, against K ∈ {1, 5, 20}.
- Silhouette against training FLOPs on one Pareto plot.
- The claim was made for graphs. Does it transfer to text?

<a id="q7"></a>
## 7. Zipfian whitening

**Question.** Does whitening the vectors with every word weighted by its frequency beat uniform centring and whitening?

**Read**
- Yokoi, Bao, Kurita & Shimodaira, *Zipfian Whitening*, NeurIPS 2024 — [arXiv:2411.00680](https://arxiv.org/abs/2411.00680)

**Measure**
- One trained table, post-processed four ways: none, centring, uniform whitening, Zipfian whitening.
- Silhouette and the nearest neighbours of the representative words.
- No training cost: report the post-processing FLOPs only.

<a id="q8"></a>
## 8. A linear gender direction

**Question.** Is gender a linear direction in the *Flatland* vectors — and in a book where women are lines, which words lie along it?

**Read**
- Park, Choe & Veitch, *The Linear Representation Hypothesis and the Geometry of Large Language Models*, ICML 2024 — [arXiv:2311.03658](https://arxiv.org/abs/2311.03658)
- Jiang, Rajendran, Ravikumar, Aragam & Veitch, *On the Origins of Linear Representations in Large Language Models*, ICML 2024 — [arXiv:2403.03867](https://arxiv.org/abs/2403.03867)

**Measure**
- Directions from man–woman, men–women, husband–wife; their pairwise cosines.
- Project the vocabulary on the mean direction: which words sit at the extremes?
- Stability over five seeds, and at D = 8 vs. D = 64.

---

## What you hand in

A report of at most eight pages, plus the code behind every number in it.

1. **Question** — which one, and how you read it.
2. **Literature** — what the papers predict for *this* setting.
3. **Hypothesis** — one falsifiable sentence, written before the experiments.
4. **Implementation** — files, how to run them, gradient checks for new ops.
5. **Protocol** — what varied, what was fixed, seeds, budget.
6. **Results** — the `summarize()` table with the baseline, and the `pareto()` plot.
7. **Threats to validity** — what could be carrying the result instead of your idea.
8. **Open** — the next experiment, and what it would decide.
9. **Reproduction** — exact commands, seeds, wall-clock time.

| weight | assessed |
|---|---|
| **35 %** | measurement rigour: seeds, error bars, the right thing fixed, FLOPs beside every score |
| **20 %** | implementation correctness: gradient checks, a reproduced baseline |
| **15 %** | engagement with the literature: what was predicted here, and whether it happened |
| **15 %** | honesty and clarity: a clean negative result scores full marks |
| **15 %** | reproducibility: someone else gets your table from your repository |

A single seed, a score without its FLOPs, or a claim smaller than its error bar fails the
project.